# hybridquant: Colab setup + smoke test

런타임 > 런타임 유형 변경 > **L4** 또는 **A100** 선택 후 실행.

비공개 레포라면 왼쪽 사이드바 열쇠 아이콘(Secrets)에 `GH_TOKEN` (GitHub Personal Access Token, repo 권한) 저장 후 노트북 접근 허용.

In [ ]:
import os, subprocess
REPO = "skytinstone/hybridquant"
BRANCH = "main"
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = os.environ.get("GH_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists("hybridquant"):
    subprocess.run(["git", "clone", "-b", BRANCH, url, "hybridquant"], check=True)
%cd hybridquant
!git log --oneline -3

In [ ]:
# 의존성. transformers는 main 브랜치 필요 (qwen3_5 모델 지원). 약 3~5분.
!pip install -q -U "transformers @ git+https://github.com/huggingface/transformers.git@main" accelerate datasets
!pip install -q flash-linear-attention kernels causal-conv1d
!pip install -q -e .
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

In [ ]:
from hybridquant import load_model, HybridQuantConfig, decode_nll, QuantStats, quantized_cache, make_cache
MODEL = "Qwen/Qwen3.5-0.8B"   # 0.8B로 파이프라인 검증 후 2B/4B/9B
model, tok = load_model(MODEL)
print(model.config.layer_types if hasattr(model.config, "layer_types") else getattr(model.config, "text_config", None))
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# 1) 훅이 실제로 호출되는지 확인
text = "The theory of general relativity describes gravity as a geometric property of spacetime. " * 40
ids = tok(text, return_tensors="pt")["input_ids"][:, :512]
for kv, st in [(16, 16), (4, 8), (2, 4)]:
    cfg = HybridQuantConfig.uniform(kv, st)
    r = decode_nll(model, ids, prefill_len=256, cfg=cfg, max_decode=128)
    print(f"{cfg.label():26s} decode-PPL={r.ppl:8.3f} | {r.stats}")

기대 결과: `attn_layers`에 어텐션 레이어 인덱스(0.8B 기준 4의 배수 근처), `gdn_layers`에 나머지가 찍혀야 함. fp(16/16) 대비 (2/4)에서 PPL이 눈에 띄게 올라야 훅이 작동하는 것.

In [ ]:
# 2) 오차-위치 곡선: state 오차가 decode 위치에 따라 누적되는지
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
for kv, st in [(16,16), (16,4), (4,16), (4,4)]:
    r = decode_nll(model, ids, 256, HybridQuantConfig.uniform(kv, st), max_decode=256)
    import numpy as np
    curve = np.convolve(r.nll_per_pos, np.ones(16)/16, mode="valid")
    plt.plot(curve, label=r.label)
plt.xlabel("decode position"); plt.ylabel("NLL (moving avg 16)"); plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
# 3) 소규모 그리드 (WikiText-2, 4 docs). L4에서 0.8B 기준 약 5~8분.
!python scripts/run_grid.py --model $MODEL --kv-bits 16 8 4 2 --state-bits 16 8 4 --n-docs 4 --prefill 256 --decode 256 --out results/smoke_0.8b.jsonl
import pandas as pd, json
df = pd.DataFrame([json.loads(l) for l in open("results/smoke_0.8b.jsonl")])
print(df.pivot(index="kv_bits", columns="state_bits", values="mean_nll").round(4))

In [ ]:
# 4) 결과를 레포에 커밋 (토큰 필요). 또는 Drive로 복사.
!git config user.email "horcruxtech.ai@gmail.com" && git config user.name "skytinstone"
!git add results/*.jsonl && git commit -m "smoke: 0.8B grid on colab" && git push